# 04 — Iran 1979 Revolution: sensitivity analysis (ORIGINAL VALIDATED INPUT ONLY)

This notebook is a follow-up to `03_nlp_analysis.ipynb` and **does not fit TF-IDF or NMF**. It reads **only** the user-generated, source-validated file `03_original_reference_annual_shares_VALIDATED.csv` from the existing project directory. The previously assistant-generated `annual_speech_topic_shares_1972_2025.csv` is **not used**.

**Run instructions:** Keep the verified CSV where you moved it (publication archive / research support). Place this notebook plus the unmodified original `Supplementary_Table_S1_All_18_Contrasts.csv` in your `notebooks/` directory. Select **Run → Run All Cells**. The first code cell locates the CSV; the inferential code confirms all 18 baseline effects against your *original publication table* (within its two-decimal reporting precision) **before saving sensitivity outputs**.

If auto-discovery finds multiple files with the same name, it stops; enter the exact path in `INPUT_FILE_OVERRIDE` to resolve. All new outputs go to `04_iran_revolution_sensitivity_outputs/` beside the notebook.


## 1. Load the actual Notebook 03 export; validate every speech-year and the source path


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from scipy.stats import norm

# IMPORTANT: Only the CSV exported by your own original reference-recovery notebook.
# Set an exact path below only if auto-detection cannot find the file or finds duplicates.
# Example: INPUT_FILE_OVERRIDE = Path.home() / 'Projects/7-masoud/results/publication/archive/research_support/03_original_reference_annual_shares_VALIDATED.csv'
INPUT_FILE_OVERRIDE = None
SOURCE_FILENAME = '03_original_reference_annual_shares_VALIDATED.csv'

PROJECT_ROOT = Path.home() / 'Projects' / '7-masoud'
WORKDIR = Path.cwd()

if INPUT_FILE_OVERRIDE is not None:
    INPUT_FILE = Path(INPUT_FILE_OVERRIDE).expanduser().resolve()
else:
    # Detect the notebook working folder and main project; no downloads or new source data.
    roots = []
    for root in [WORKDIR, *WORKDIR.parents, PROJECT_ROOT]:
        if root.is_dir() and root not in roots and (root == WORKDIR or (root / 'notebooks').is_dir()):
            roots.append(root)
    direct = []
    suggested_subfolders = [
        '', 'publication_archive/research_support',
        'publication/archive/research_support',
        'results/publication/archive/research_support',
        'results/publication/research_support',
        'results/publication/tables',
        'notebooks',
    ]
    for root in roots:
        for sub in suggested_subfolders:
            p = root / sub / SOURCE_FILENAME
            if p.is_file(): direct.append(p.resolve())
    matches = list(dict.fromkeys(direct))
    if not matches:
        for root in roots:
            if root.is_dir():
                matches.extend(p.resolve() for p in root.rglob(SOURCE_FILENAME))
        matches=list(dict.fromkeys(matches))
    if len(matches) != 1:
        raise RuntimeError(
            f'Expected exactly ONE original validated CSV. Found {len(matches)}: {matches}. '
            'Set INPUT_FILE_OVERRIDE to the exact path in the first cell.'
        )
    INPUT_FILE = matches[0]

if not INPUT_FILE.is_file():
    raise FileNotFoundError(f'The ORIGINAL validated file is missing: {INPUT_FILE}')
if INPUT_FILE.name != SOURCE_FILENAME:
    raise AssertionError('STOP: Wrong input filename. Only the original validated CSV is allowed.')

data = pd.read_csv(INPUT_FILE)
YEARS=np.arange(1972,2026)
COUNTRIES=['IRN','USA','CHN']
TOPIC_COLS=[f'T{i:02d}' for i in range(1,8)]
ELIGIBLE=[1,2,3,4,5,7]  # Topic 6 excluded in original paper
PAIRS=[('IRN','USA'),('IRN','CHN'),('USA','CHN')]
TOPIC_LABELS={
 1:'Economic development and global cooperation',
 2:'China and peaceful development',
 3:'Regional conflicts and diplomatic settlement (provisional)',
 4:'Iran and regional security (country-specific vocabulary)',
 5:'Human rights and freedom discourse',
 6:'Procedural language (excluded)',
 7:'Nuclear arms and disarmament',
}
required={'country_code','year','archive_path',*TOPIC_COLS}
assert required.issubset(data.columns), f'Missing required columns: {required-set(data.columns)}'
assert len(data)==162, f'Unexpected number of speeches: {len(data)}'
assert data.country_code.value_counts().to_dict()=={'IRN':54,'USA':54,'CHN':54}
assert not data.duplicated(['country_code','year']).any()
assert data[TOPIC_COLS].notna().all().all()
assert np.isfinite(data[TOPIC_COLS].to_numpy(dtype=float)).all()
assert data[TOPIC_COLS].to_numpy(dtype=float).min()>=-1e-10
assert np.allclose(data[TOPIC_COLS].sum(axis=1).values,1.0,atol=1e-8)
series={}
for country in COUNTRIES:
    tmp=data.loc[data.country_code==country].sort_values('year')
    assert np.array_equal(tmp.year.to_numpy(), YEARS), f'Incorrect years for {country}'
    series[country]=tmp[TOPIC_COLS].to_numpy(dtype=float)

print('ORIGINAL VALIDATED INPUT:', INPUT_FILE)
print('Verified:', len(data), 'speech-years / 54 per country / 7 topics.')
print('No TF-IDF or NMF training performed.')


## 2. Specify historical periods and the original estimator

In [ ]:
# Three original time periods: early (1972–90), middle (1991–2010), late (2011–25).
D_original = np.column_stack([
    np.ones(len(YEARS)),
    ((YEARS>=1991)&(YEARS<=2010)).astype(float),
    (YEARS>=2011).astype(float),
])

# Four periods separating 1979: pre, post, middle, late.
period = np.select(
    [YEARS<=1978, YEARS<=1990, YEARS<=2010],
    ['pre_1979','post_1979_1990','middle_1991_2010'],
    default='late_2011_2025'
)
PERIODS=['pre_1979','post_1979_1990','middle_1991_2010','late_2011_2025']
D_four=np.column_stack([np.ones(len(YEARS))]+[(period==p).astype(float) for p in PERIODS[1:]])

# Optional exclusion of transition years 1979–1981 from post-period reference.
transition=np.select(
 [YEARS<=1978,YEARS<=1981,YEARS<=1990,YEARS<=2010],
 ['pre','transition_1979_81','post_1982_1990','middle'], default='late'
)
D_transition=np.column_stack([
 np.ones(len(YEARS))]+[(transition==p).astype(float) for p in
 ['transition_1979_81','post_1982_1990','middle','late']]
)

SCENARIOS={
 'original_late_minus_1972_1990':(D_original, np.array([0,0,1])),
 'late_minus_pre1979':(D_four,np.array([0,0,0,1])),
 'late_minus_post1979_1990':(D_four,np.array([0,-1,0,1])),
 'post1979_1990_minus_pre1979':(D_four,np.array([0,1,0,0])),
 'late_minus_post1982_1990_excluding_transition':(D_transition,np.array([0,0,-1,0,1])),
}

def hac_contrast(y, design, contrast, lag=5):
    model=sm.OLS(y,design).fit(cov_type='HAC',cov_kwds={'maxlags':lag,'use_correction':True})
    effect=float(contrast @ model.params)
    se=float(np.sqrt(contrast @ model.cov_params() @ contrast))
    z=norm.ppf(0.975)
    p=float(2*norm.sf(abs(effect/se))) if se>0 else np.nan
    return dict(effect_pp=effect,hac_se_pp=se,ci_lower_pp=effect-z*se,
                ci_upper_pp=effect+z*se,p_value=p)

print('Scenario definitions:', list(SCENARIOS))

## 3. Check all 18 original results first; then save the new sensitivity comparisons


In [ ]:
def run_bilateral(lag=5):
    rows=[]
    for a,b in PAIRS:
        for t in ELIGIBLE:
            y=100*(series[a][:,t-1]-series[b][:,t-1])
            for scenario,(d,c) in SCENARIOS.items():
                rows.append({'pair':f'{a}-{b}','topic_id':t,
                             'topic_label':TOPIC_LABELS[t],'scenario':scenario,
                             'hac_lag':lag,**hac_contrast(y,d,c,lag)})
    result=pd.DataFrame(rows)
    for scenario,indices in result.groupby('scenario').groups.items():
        result.loc[indices,'holm_adjusted_p_18']=multipletests(
            result.loc[indices,'p_value'],method='holm')[1]
    result['holm_significant_18']=result.holm_adjusted_p_18<.05
    return result

bilateral = run_bilateral(lag=5)

# Use ONLY the user-provided publication benchmark, unchanged.
REFERENCE_NAME='Supplementary_Table_S1_All_18_Contrasts.csv'
REFERENCE_OVERRIDE=None  # Set the exact path only if file auto-discovery fails.
if REFERENCE_OVERRIDE is not None:
    REFERENCE_FILE=Path(REFERENCE_OVERRIDE).expanduser().resolve()
else:
    ref_candidates=[WORKDIR/REFERENCE_NAME,
                    PROJECT_ROOT/'results/publication/tables'/REFERENCE_NAME,
                    PROJECT_ROOT/'publication_archive/research_support'/REFERENCE_NAME,
                    PROJECT_ROOT/'results/publication/archive/research_support'/REFERENCE_NAME]
    ref_matches=list(dict.fromkeys(p.resolve() for p in ref_candidates if p.is_file()))
    if not ref_matches:
        raise FileNotFoundError('Put the UNCHANGED original Supplementary_Table_S1_All_18_Contrasts.csv beside this notebook, or set REFERENCE_OVERRIDE.')
    if len(ref_matches)>1:
        raise RuntimeError(f'Multiple original reference files found: {ref_matches}. Set REFERENCE_OVERRIDE to select one.')
    REFERENCE_FILE=ref_matches[0]
ref=pd.read_csv(REFERENCE_FILE,encoding='utf-8-sig')
assert {'Country contrast','Topic','Effect (pp)'}.issubset(ref.columns)
assert len(ref)==18 and not ref.duplicated(['Country contrast','Topic']).any()
pair_label_to_code={'Iran - China':'IRN-CHN',
                    'Iran - United States':'IRN-USA',
                    'United States - China':'USA-CHN'}
check=[]
for _,r in ref.iterrows():
    pair=pair_label_to_code[str(r['Country contrast']).strip()]
    t=int(r['Topic'])
    baseline=bilateral.query("scenario=='original_late_minus_1972_1990' and pair==@pair and topic_id==@t")
    assert len(baseline)==1
    original=float(r['Effect (pp)'])
    observed=float(baseline.iloc[0].effect_pp)
    check.append({'pair':pair,'topic_id':t,
                  'source_publication_effect_pp':original,
                  'recomputed_effect_pp':observed,
                  'absolute_difference_pp':abs(observed-original)})
check=pd.DataFrame(check)
assert len(check)==18
max_discrepancy=float(check.absolute_difference_pp.max())
print('Original publication benchmark:',REFERENCE_FILE)
print('Maximum discrepancy across ALL 18 contrasts (pp):',round(max_discrepancy,8))
# Publication table reports two decimals; rounding tolerance = half of 0.01 pp.
assert max_discrepancy <= 0.0051, 'STOP: original validated CSV did not reproduce all 18 published effects. Nothing saved.'
print('PASSED: ALL 18 historical contrasts match the ORIGINAL table (two-decimal rounding).')

# No output files are created until the complete validation above passes.
OUT = WORKDIR / '04_iran_revolution_sensitivity_outputs'
OUT.mkdir(exist_ok=True)
check.to_csv(OUT/'original_all_18_validation.csv',index=False)
bilateral.to_csv(OUT/'bilateral_all_18_by_scenario.csv',index=False)
print('Input source recorded as:', INPUT_FILE)
display(bilateral.query("pair=='IRN-USA' and topic_id==7")[[
    'scenario','effect_pp','ci_lower_pp','ci_upper_pp',
    'holm_adjusted_p_18','holm_significant_18'
]].round(5))


## 4. Direct changes within Iran and means for interpretation

In [ ]:
iran_rows=[]
for t in ELIGIBLE:
    y=100*series['IRN'][:,t-1]
    d,c=SCENARIOS['post1979_1990_minus_pre1979']
    iran_rows.append({'topic_id':t,'topic_label':TOPIC_LABELS[t],**hac_contrast(y,d,c,lag=5)})
iran=pd.DataFrame(iran_rows)
iran['holm_adjusted_p_6']=multipletests(iran.p_value,method='holm')[1]
iran['holm_significant_6']=iran.holm_adjusted_p_6<.05
iran.to_csv(OUT/'iran_pre_post_changes.csv',index=False)

means=[]
for country in COUNTRIES:
    for t in range(1,8):
        for per in PERIODS:
            values=series[country][period==per,t-1]*100
            means.append({'country_code':country,'topic_id':t,'period':per,
                          'n_years':len(values),'mean_share_pct':values.mean(),
                          'median_share_pct':np.median(values)})
means=pd.DataFrame(means)
means.to_csv(OUT/'topic_means_by_country_period.csv',index=False)
print('Direct changes within Iran (1979–1990 minus 1972–1978):')
display(iran[['topic_id','topic_label','effect_pp','holm_adjusted_p_6','holm_significant_6']].round(5))

## 5. Robustness: change the Newey–West bandwidth (3, 5, 10)

In [ ]:
hac_lags=pd.concat([run_bilateral(lag=l) for l in [3,5,10]],ignore_index=True)
hac_lags.to_csv(OUT/'HAC_lag_3_5_10.csv',index=False)
display(hac_lags.query("pair=='IRN-USA' and topic_id==7")[[
 'hac_lag','scenario','effect_pp','holm_adjusted_p_18','holm_significant_18'
]].round(5))

## 6. Publication-friendly figures, using the saved topic shares

In [ ]:
scen_order=['original_late_minus_1972_1990',
            'late_minus_pre1979',
            'late_minus_post1979_1990',
            'late_minus_post1982_1990_excluding_transition']
scen_names=['Original: 1972–90', 'Pre-revolution: 1972–78',
            'Post-revolution: 1979–90','Post-transition: 1982–90']
fig,ax=plt.subplots(figsize=(9.5,5.2))
for i,scen in enumerate(scen_order):
    r=bilateral.query('pair=="IRN-USA" and topic_id==7 and scenario==@scen').iloc[0]
    errs=np.array([[r.effect_pp-r.ci_lower_pp],[r.ci_upper_pp-r.effect_pp]])
    ax.errorbar(r.effect_pp,i,xerr=errs,fmt='o',color='black',capsize=4)
ax.axvline(0,color='gray',ls='--',lw=1)
ax.set_yticks(range(4),scen_names)
ax.invert_yaxis()
ax.set_xlabel('Change in Iran–US T07 topic-share gap (percentage points)')
ax.set_title('Iran 1979 sensitivity — nuclear arms and disarmament')
ax.grid(axis='x',alpha=0.2)
fig.tight_layout()
fig.savefig(OUT/'Iran_US_T07_Revolution_Sensitivity.png',dpi=220)
plt.show()

fig,axes=plt.subplots(2,1,figsize=(11,7),sharex=True)
for ax,t in zip(axes,[3,7]):
    for country in COUNTRIES:
        ax.plot(YEARS,100*series[country][:,t-1],lw=1.3,label=country)
    ax.axvline(1979,color='black',ls='--',lw=1.2)
    ax.set_ylabel('Topic share (%)')
    ax.set_title(f'T{t:02d}: {TOPIC_LABELS[t]}')
    ax.grid(axis='y',alpha=.2)
axes[0].legend(ncol=3)
axes[-1].set_xlabel('Year')
fig.tight_layout()
fig.savefig(OUT/'Historical_T03_T07_Topic_Shares.png',dpi=200)
plt.show()
print('All results saved in:',OUT.resolve())

## 7. Interpretation / restrictions

- **Source:** only `03_original_reference_annual_shares_VALIDATED.csv` generated from the user's original processed corpus and the reference model used in Notebook 03.
- The original period 1972–1990 combines pre- and post-revolution observations; this notebook redefines reference periods while keeping the original topic-score matrix fixed.
- The analysis is **descriptive sensitivity**, not a causal interrupted-time-series estimate of the revolution effect.
- The pre-1979 period (1972–1978) has seven speeches per country: interpret confidence intervals cautiously.
- After validating against all 18 original contrasts (and only then), it writes additional statistical summaries and figures beside the notebook.
- Never replace the validated CSV with an assistant-generated alternative. Labels remain provisional and NMF proportions are compositional.
- In the manuscript, distinguish the 1972–1978 versus 1979–1990 comparison from the original 1972–1990 versus 2011–2025 contrast.
